In [0]:
DROP CATALOG IF EXISTS shoppulse CASCADE;

CREATE CATALOG IF NOT EXISTS shoppulse;
CREATE SCHEMA IF NOT EXISTS shoppulse.dev;
CREATE VOLUME IF NOT EXISTS shoppulse.dev.raw;



SELECT status, count(*) AS nb
FROM read_files('/Volumes/shoppulse/dev/raw/landing', format => 'json')
GROUP BY status;

SELECT count(*), min(ingestion_timestamp), max(ingestion_timestamp)
FROM shoppulse.dev.shoppulse_bronze;

DESCRIBE DETAIL shoppulse.dev.shoppulse_bronze;

DROP TABLE IF EXISTS shoppulse.dev.shoppulse_bronze;
CREATE VOLUME IF NOT EXISTS shoppulse.dev.checkpoints;

SELECT count(*) FROM shoppulse.dev.shoppulse_bronze;
SELECT * FROM shoppulse.dev.shoppulse_bronze;


SELECT count(*) FROM shoppulse.dev.shoppulse_bronze WHERE _rescued_data IS NOT NULL;



CREATE TABLE IF NOT EXISTS shoppulse.dev.shoppulse_silver (
  transaction_id      STRING,
  user_id             STRING,
  product_id          STRING,
  amount              DOUBLE,
  transaction_date    TIMESTAMP,
  status              STRING,
  ingestion_timestamp TIMESTAMP
);


DESCRIBE HISTORY shoppulse.dev.shoppulse_silver


CREATE OR REPLACE VIEW shoppulse.dev.v_user_metrics AS
SELECT
  user_id,
  annee_mois,
  total_amount AS chiffre_affaires,
  avg_basket   AS panier_moyen,
  nb_orders    AS nb_commandes
FROM shoppulse.dev.shoppulse_gold_user_metrics;


SELECT 
annee_mois,
SUM(chiffre_affaires) AS total_ca,
user_id 
FROM shoppulse.dev.v_user_metrics
GROUP BY annee_mois,user_id
ORDER BY total_ca 
LIMIT 10;

SELECT user_id,
       round(sum(chiffre_affaires), 2) AS ca_total,
       sum(nb_commandes)               AS nb_commandes
FROM shoppulse.dev.v_user_metrics
GROUP BY user_id
ORDER BY ca_total DESC
LIMIT 10;

SELECT 
annee_mois,
SUM(chiffre_affaires) AS total_ca
FROM shoppulse.dev.v_user_metrics
GROUP BY annee_mois
ORDER BY annee_mois 

SELECT annee_mois,sum(nb_commandes) AS nb_commandes
FROM shoppulse.dev.v_user_metrics
group by annee_mois
ORDER BY annee_mois ASC


SELECT 'bronze' AS couche, count(*) AS nb FROM shoppulse.prod.shoppulse_bronze
UNION ALL SELECT 'silver', count(*) FROM shoppulse.prod.shoppulse_silver
UNION ALL SELECT 'gold',   count(*) FROM shoppulse.prod.shoppulse_gold_user_metrics;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.